<a href="https://colab.research.google.com/github/ThayseBachmeyer/ProcessamentoSinaisI/blob/main/Aula%2007/C%C3%B3digos/prat7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Código utilizado na questão 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 20000
valores_r = [0.5, 0.85, 0.98]

projetos = [
    ("Passa-baixas", 1000),
    ("Passa-altas", 2000),
    ("Passa-faixas", 7000),
    ("Notch", 3000)
]

def projetar_filtro_2ordem(tipo, fc, fs, r):

    theta_c = 2 * np.pi * fc / fs

    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])

    if tipo == "Passa-baixas":

        b = np.array([1.0, 2.0, 1.0])

        ganho_dc = np.sum(b) / np.sum(a)
        b = b / ganho_dc

    elif tipo == "Passa-altas":

        b = np.array([1.0, -2.0, 1.0])

        ganho_nyq = (b[0] - b[1] + b[2]) / (a[0] - a[1] + a[2])
        b = b / ganho_nyq

    elif tipo == "Passa-faixas":

        b = np.array([1.0, 0.0, -1.0])

        z_c = np.exp(1j * theta_c)
        H_zc = (b[0] + b[1]*z_c**(-1) + b[2]*z_c**(-2)) / (a[0] + a[1]*z_c**(-1) + a[2]*z_c**(-2))
        b = b / np.abs(H_zc)

    elif tipo == "Notch":

        b = np.array([1.0, -2 * np.cos(theta_c), 1.0])

        ganho_dc = np.sum(b) / np.sum(a)
        b = b / ganho_dc

    return b, a

for tipo, fc in projetos:
    print("=" * 75)
    print(f"PROJETO: {tipo.upper()} | fc = {fc} Hz | fs = {fs} Hz")
    print("=" * 75)

    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Filtro {tipo} de 2ª Ordem ($f_c = {fc}$ Hz, $f_s = {fs}$ Hz)', fontsize=14, fontweight='bold')

    angulo_circ = np.linspace(0, 2 * np.pi, 400)
    axs[1].plot(np.cos(angulo_circ), np.sin(angulo_circ), 'k--', alpha=0.5, label='Círculo Unitário')
    axs[1].axhline(0, color='gray', lw=0.8)
    axs[1].axvline(0, color='gray', lw=0.8)

    cores = ['tab:blue', 'tab:orange', 'tab:green']

    for idx, r in enumerate(valores_r):
        b, a = projetar_filtro_2ordem(tipo, fc, fs, r)

        print(f"\n-> Para r = {r}:")
        print(f"   Numerador B(z)   = {b[0]:.4f} + ({b[1]:.4f})z^-1 + ({b[2]:.4f})z^-2")
        print(f"   Denominador A(z) = {a[0]:.4f} + ({a[1]:.4f})z^-1 + ({a[2]:.4f})z^-2")

        w, h = signal.freqz(b, a, worN=4096, fs=fs)
        magnitude_db = 20 * np.log10(np.maximum(np.abs(h), 1e-10))
        axs[0].plot(w, magnitude_db, color=cores[idx], lw=2, label=f'r = {r}')

        zeros = np.roots(b)
        polos = np.roots(a)

        if idx == 0:
            axs[1].scatter(np.real(zeros), np.imag(zeros), s=90, facecolors='none', edgecolors='red', lw=2, label='Zeros')

        axs[1].scatter(np.real(polos), np.imag(polos), s=90, marker='x', color=cores[idx], lw=2, label=f'Polos (r={r})')

    axs[0].axvline(fc, color='red', linestyle=':', alpha=0.7, label=f'$f_c = {fc}$ Hz')
    axs[0].set_title('Resposta em Frequência (Magnitude)')
    axs[0].set_xlabel('Frequência (Hz)')
    axs[0].set_ylabel('Magnitude (dB)')
    axs[0].set_ylim(-60, 15)
    axs[0].set_xlim(0, fs / 2)
    axs[0].grid(True, alpha=0.4)
    axs[0].legend()

    axs[1].set_title('Diagrama de Polos e Zeros (Plano Z)')
    axs[1].set_xlabel('Parte Real')
    axs[1].set_ylabel('Parte Imaginária')
    axs[1].set_xlim(-1.3, 1.3)
    axs[1].set_ylim(-1.3, 1.3)
    axs[1].set_aspect('equal', adjustable='box')
    axs[1].grid(True, alpha=0.4)
    axs[1].legend(loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.show()

##Código utilizado na questão 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 20000
fc1 = 6000
fc2 = 8000
fc_q1 = 7000
r = 0.90

def bloco_passa_faixas_2ordem(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs

    b = np.array([1.0, 0.0, -1.0])

    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])

    z_c = np.exp(1j * theta_c)
    H_zc = (b[0] + b[1]*z_c**(-1) + b[2]*z_c**(-2)) / (a[0] + a[1]*z_c**(-1) + a[2]*z_c**(-2))
    b = b / np.abs(H_zc)
    return b, a

b_q1, a_q1 = bloco_passa_faixas_2ordem(fc_q1, fs, r)

b1, a1 = bloco_passa_faixas_2ordem(fc1, fs, r)
b2, a2 = bloco_passa_faixas_2ordem(fc2, fs, r)

b_cascata = np.convolve(b1, b2)
a_cascata = np.convolve(a1, a2)

w, h_cascata_bruto = signal.freqz(b_cascata, a_cascata, worN=4096, fs=fs)
ganho_max_cascata = np.max(np.abs(h_cascata_bruto))
b_cascata = b_cascata / ganho_max_cascata

print("=" * 75)
print(f"FILTRO PASSA-FAIXAS EM CASCATA (4ª ORDEM) | r = {r}")
print("=" * 75)
print("Numerador B(z) (potências de z^0 a z^-4):")
print(np.round(b_cascata, 5))
print("\nDenominador A(z) (potências de z^0 a z^-4):")
print(np.round(a_cascata, 5))

_, h_q1 = signal.freqz(b_q1, a_q1, worN=4096, fs=fs)
_, h_cascata = signal.freqz(b_cascata, a_cascata, worN=4096, fs=fs)

mag_q1_db = 20 * np.log10(np.maximum(np.abs(h_q1), 1e-10))
mag_cascata_db = 20 * np.log10(np.maximum(np.abs(h_cascata), 1e-10))

fig, axs = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Comparação: Filtro Cascata (6-8 kHz) vs. Filtro Q1 (7 kHz) | $r = {r}$', fontsize=14, fontweight='bold')

axs[0].plot(w, mag_cascata_db, color='tab:blue', lw=2.5, label='Questão 2: Cascata ($f_{c1}=6$ kHz, $f_{c2}=8$ kHz)')
axs[0].plot(w, mag_q1_db, color='tab:orange', lw=2, linestyle='--', label='Questão 1: Bloco Único ($f_c=7$ kHz)')
axs[0].axhline(-3, color='red', linestyle=':', alpha=0.8, label='Limite -3 dB (Largura de Banda)')
axs[0].axvline(6000, color='gray', linestyle='-.', alpha=0.5)
axs[0].axvline(8000, color='gray', linestyle='-.', alpha=0.5)

axs[0].set_title('Resposta em Frequência (Magnitude em dB)')
axs[0].set_xlabel('Frequência (Hz)')
axs[0].set_ylabel('Magnitude (dB)')
axs[0].set_ylim(-60, 5)
axs[0].set_xlim(0, fs / 2)
axs[0].grid(True, alpha=0.4)
axs[0].legend()

angulo = np.linspace(0, 2 * np.pi, 400)
axs[1].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5, label='Círculo Unitário')
axs[1].axhline(0, color='gray', lw=0.8)
axs[1].axvline(0, color='gray', lw=0.8)

z_casc = np.roots(b_cascata)
p_casc = np.roots(a_cascata)
axs[1].scatter(np.real(z_casc), np.imag(z_casc), s=110, facecolors='none', edgecolors='blue', lw=2, label='Zeros Cascata (Duplos em ±1)')
axs[1].scatter(np.real(p_casc), np.imag(p_casc), s=100, marker='x', color='blue', lw=2.5, label='Polos Cascata (6 kHz e 8 kHz)')

z_q1 = np.roots(b_q1)
p_q1 = np.roots(a_q1)
axs[1].scatter(np.real(p_q1), np.imag(p_q1), s=90, marker='+', color='orange', lw=2, label='Polos Q1 (7 kHz)')

axs[1].set_title('Diagrama de Polos e Zeros (Plano Z)')
axs[1].set_xlabel('Parte Real')
axs[1].set_ylabel('Parte Imaginária')
axs[1].set_xlim(-1.3, 1.3)
axs[1].set_ylim(-1.3, 1.3)
axs[1].set_aspect('equal', adjustable='box')
axs[1].grid(True, alpha=0.4)
axs[1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()

##Código utilizado na questão 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 20000
fc1 = 1000
fc2 = 4000
fc_notch = 3000
r = 0.85

def bloco_passa_baixas_2ordem(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    ganho_dc = np.sum(b) / np.sum(a)
    return b / ganho_dc, a

def bloco_passa_altas_2ordem(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    ganho_nyq = (b[0] - b[1] + b[2]) / (a[0] - a[1] + a[2])
    return b / ganho_nyq, a

def bloco_notch_2ordem(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    ganho_dc = np.sum(b) / np.sum(a)
    return b / ganho_dc, a

b_pb, a_pb = bloco_passa_baixas_2ordem(fc1, fs, r)
b_pa, a_pa = bloco_passa_altas_2ordem(fc2, fs, r)

b_paralelo = np.convolve(b_pb, a_pa) + np.convolve(b_pa, a_pb)
a_paralelo = np.convolve(a_pb, a_pa)

b_notch, a_notch = bloco_notch_2ordem(fc_notch, fs, r)

print("=" * 75)
print(f"FILTRO REJEITA-FAIXAS EM PARALELO (4ª ORDEM) | r = {r}")
print("=" * 75)
print("Numerador B(z) (potências de z^0 a z^-4):")
print(np.round(b_paralelo, 5))
print("\nDenominador A(z) (potências de z^0 a z^-4):")
print(np.round(a_paralelo, 5))

w, h_paralelo = signal.freqz(b_paralelo, a_paralelo, worN=4096, fs=fs)
_, h_notch = signal.freqz(b_notch, a_notch, worN=4096, fs=fs)

mag_par_db = 20 * np.log10(np.maximum(np.abs(h_paralelo), 1e-10))
mag_notch_db = 20 * np.log10(np.maximum(np.abs(h_notch), 1e-10))

fig, axs = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Comparação: Rejeita-Faixas Paralelo (1-4 kHz) vs. Notch Q1 (3 kHz) | $r = {r}$', fontsize=14, fontweight='bold')

axs[0].plot(w, mag_par_db, color='tab:blue', lw=2.5, label='Questão 3: Paralelo PB(1 kHz) + PA(4 kHz)')
axs[0].plot(w, mag_notch_db, color='tab:orange', lw=2, linestyle='--', label='Questão 1: Notch Simples ($f_c=3$ kHz)')
axs[0].axhline(-3, color='red', linestyle=':', alpha=0.8, label='Limite -3 dB')
axs[0].axvline(1000, color='gray', linestyle='-.', alpha=0.5)
axs[0].axvline(4000, color='gray', linestyle='-.', alpha=0.5)

axs[0].set_title('Resposta em Frequência (Magnitude em dB)')
axs[0].set_xlabel('Frequência (Hz)')
axs[0].set_ylabel('Magnitude (dB)')
axs[0].set_ylim(-60, 10)
axs[0].set_xlim(0, fs / 2)
axs[0].grid(True, alpha=0.4)
axs[0].legend()

angulo = np.linspace(0, 2 * np.pi, 400)
axs[1].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5, label='Círculo Unitário')
axs[1].axhline(0, color='gray', lw=0.8)
axs[1].axvline(0, color='gray', lw=0.8)

z_par = np.roots(b_paralelo)
p_par = np.roots(a_paralelo)
axs[1].scatter(np.real(z_par), np.imag(z_par), s=110, facecolors='none', edgecolors='blue', lw=2, label='Zeros Paralelo (4ª Ordem)')
axs[1].scatter(np.real(p_par), np.imag(p_par), s=100, marker='x', color='blue', lw=2.5, label='Polos Paralelo (1 kHz e 4 kHz)')

z_n = np.roots(b_notch)
p_n = np.roots(a_notch)
axs[1].scatter(np.real(z_n), np.imag(z_n), s=90, facecolors='none', edgecolors='orange', lw=2, label='Zeros Notch Q1 (3 kHz)')
axs[1].scatter(np.real(p_n), np.imag(p_n), s=90, marker='+', color='orange', lw=2, label='Polos Notch Q1 (3 kHz)')

axs[1].set_title('Diagrama de Polos e Zeros (Plano Z)')
axs[1].set_xlabel('Parte Real')
axs[1].set_ylabel('Parte Imaginária')
axs[1].set_xlim(-1.3, 1.3)
axs[1].set_ylim(-1.3, 1.3)
axs[1].set_aspect('equal', adjustable='box')
axs[1].grid(True, alpha=0.4)
axs[1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()

##Código utilizado na questão 4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 20000
bits_lista = [2, 4, 8, 16, 32]

def bloco_passa_faixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 0.0, -1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    z_c = np.exp(1j * theta_c)
    H_zc = (b[0] + b[1]*z_c**(-1) + b[2]*z_c**(-2)) / (a[0] + a[1]*z_c**(-1) + a[2]*z_c**(-2))
    return b / np.abs(H_zc), a

def bloco_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / (np.sum(b) / np.sum(a)), a

def bloco_passa_altas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / ((b[0] - b[1] + b[2]) / (a[0] - a[1] + a[2])), a

def quantizar_coeficientes(vetor, b):
    """
    Quantiza um vetor de coeficientes usando representação em ponto fixo
    com sinal de 'b' bits, ajustando a escala dinâmica ao maior valor do vetor.
    """
    max_val = np.max(np.abs(vetor))
    if max_val == 0:
        return vetor.copy()

    niveis = 2**(b - 1) - 1
    passo = max_val / niveis

    vetor_quant = np.round(vetor / passo) * passo
    return vetor_quant

r_q2 = 0.90
b1_q2, a1_q2 = bloco_passa_faixas(6000, fs, r_q2)
b2_q2, a2_q2 = bloco_passa_faixas(8000, fs, r_q2)

b_q2_orig = np.convolve(b1_q2, b2_q2)
a_q2_orig = np.convolve(a1_q2, a2_q2)
_, h_tmp = signal.freqz(b_q2_orig, a_q2_orig, worN=4096, fs=fs)
ganho_q2 = np.max(np.abs(h_tmp))
b_q2_orig /= ganho_q2
b1_q2 /= np.sqrt(ganho_q2)
b2_q2 /= np.sqrt(ganho_q2)

r_q3 = 0.85
b1_q3, a1_q3 = bloco_passa_baixas(1000, fs, r_q3)
b2_q3, a2_q3 = bloco_passa_altas(4000, fs, r_q3)

b_q3_orig = np.convolve(b1_q3, a2_q3) + np.convolve(b2_q3, a1_q3)
a_q3_orig = np.convolve(a1_q3, a2_q3)

def analisar_quantizacao(nome_filtro, b_orig, a_orig, blocos, topologia):
    print("=" * 85)
    print(f"ANÁLISE DE QUANTIZAÇÃO: {nome_filtro}")
    print("=" * 85)

    (b1, a1), (b2, a2) = blocos
    w, h_orig = signal.freqz(b_orig, a_orig, worN=4096, fs=fs)
    mag_orig = 20 * np.log10(np.maximum(np.abs(h_orig), 1e-10))

    fig, axs = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(f'Efeito da Quantização de Coeficientes — {nome_filtro}', fontsize=14, fontweight='bold')

    cores = ['tab:red', 'tab:orange', 'tab:green', 'tab:purple', 'tab:cyan']

    axs[0, 0].plot(w, mag_orig, 'k-', lw=2.5, label='Original (64-bit)')
    axs[0, 1].plot(w, mag_orig, 'k-', lw=2.5, label='Original (64-bit)')

    angulo = np.linspace(0, 2 * np.pi, 400)
    for col in [0, 1]:
        axs[1, col].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5)
        axs[1, col].scatter(np.real(np.roots(b_orig)), np.imag(np.roots(b_orig)), s=100, facecolors='none', edgecolors='black', lw=1.5, label='Zeros Orig.')
        axs[1, col].scatter(np.real(np.roots(a_orig)), np.imag(np.roots(a_orig)), s=100, marker='x', color='black', lw=2, label='Polos Orig.')

    print(f"{'Bits':<6} | {'Método 1 (Resultante 4ª Ordem)':<36} | {'Método 2 (Blocos de 2ª Ordem)':<36}")
    print("-" * 85)

    for idx, b in enumerate(bits_lista):

        b_res_q = quantizar_coeficientes(b_orig, b)
        a_res_q = quantizar_coeficientes(a_orig, b)

        polos_res = np.roots(a_res_q) if a_res_q[0] != 0 else np.array([np.inf])
        max_p_res = np.max(np.abs(polos_res))
        est_res = "ESTÁVEL" if max_p_res < 1.0 else "INSTÁVEL"

        _, h_res = signal.freqz(b_res_q, a_res_q, worN=4096, fs=fs)
        axs[0, 0].plot(w, 20 * np.log10(np.maximum(np.abs(h_res), 1e-10)), color=cores[idx], lw=1.5, linestyle='--', label=f'{b} bits ({est_res})')
        if np.all(np.isfinite(polos_res)):
            axs[1, 0].scatter(np.real(polos_res), np.imag(polos_res), s=70, marker='+', color=cores[idx], lw=2, label=f'Polos {b}b')

        b1_q, a1_q = quantizar_coeficientes(b1, b), quantizar_coeficientes(a1, b)
        b2_q, a2_q = quantizar_coeficientes(b2, b), quantizar_coeficientes(a2, b)

        if topologia == "cascata":
            b_blk_q = np.convolve(b1_q, b2_q)
            a_blk_q = np.convolve(a1_q, a2_q)
        else:
            b_blk_q = np.convolve(b1_q, a2_q) + np.convolve(b2_q, a1_q)
            a_blk_q = np.convolve(a1_q, a2_q)

        polos_blk = np.roots(a_blk_q) if a_blk_q[0] != 0 else np.array([np.inf])
        max_p_blk = np.max(np.abs(polos_blk))
        est_blk = "ESTÁVEL" if max_p_blk < 1.0 else "INSTÁVEL"

        _, h_blk = signal.freqz(b_blk_q, a_blk_q, worN=4096, fs=fs)
        axs[0, 1].plot(w, 20 * np.log10(np.maximum(np.abs(h_blk), 1e-10)), color=cores[idx], lw=1.5, linestyle='--', label=f'{b} bits ({est_blk})')
        if np.all(np.isfinite(polos_blk)):
            axs[1, 1].scatter(np.real(polos_blk), np.imag(polos_blk), s=70, marker='+', color=cores[idx], lw=2, label=f'Polos {b}b')

        info_res = f"{est_res} (max|p| = {max_p_res:.4f})"
        info_blk = f"{est_blk} (max|p| = {max_p_blk:.4f})"
        print(f"{b:<6} | {info_res:<36} | {info_blk:<36}")

    titulos = [
        'Resp. Frequência — Quant. Filtro Resultante',
        'Resp. Frequência — Quant. por Blocos 2ª Ordem',
        'Polos e Zeros — Quant. Filtro Resultante',
        'Polos e Zeros — Quant. por Blocos 2ª Ordem'
    ]
    for i, ax in enumerate(axs.flat):
        ax.set_title(titulos[i])
        ax.grid(True, alpha=0.4)
        ax.legend(fontsize=8, loc='best')
        if i < 2:
            ax.set_xlabel('Frequência (Hz)')
            ax.set_ylabel('Magnitude (dB)')
            ax.set_ylim(-65, 15)
            ax.set_xlim(0, fs / 2)
        else:
            ax.set_xlabel('Parte Real')
            ax.set_ylabel('Parte Imaginária')
            ax.set_xlim(-1.4, 1.4)
            ax.set_ylim(-1.4, 1.4)
            ax.set_aspect('equal', adjustable='box')

    plt.tight_layout()
    plt.show()

analisar_quantizacao("Questão 2: Passa-Faixas em Cascata", b_q2_orig, a_q2_orig, ((b1_q2, a1_q2), (b2_q2, a2_q2)), "cascata")
analisar_quantizacao("Questão 3: Rejeita-Faixas em Paralelo", b_q3_orig, a_q3_orig, ((b1_q3, a1_q3), (b2_q3, a2_q3)), "paralelo")

##Código utilizado na questão 5-a

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)

if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64)
x = x / np.max(np.abs(x))

N = len(x)
t = np.arange(N) / fs

print(f"Frequência de amostragem (fs): {fs} Hz | Total de amostras: {N} | Duração: {N/fs:.2f} s")

interf_baixa = 0.05 * np.cos(200 * np.pi * t)
interf_alta = 0.075 * np.sin(4000 * np.pi * t)

variancias = [1e-2, 1e-1, 1.0]
sinais_contaminados = {}

np.random.seed(42)

for var in variancias:
    desvio_padrao = np.sqrt(var)
    ruido_branco = np.random.normal(0, desvio_padrao, N)
    y = x + interf_baixa + interf_alta + ruido_branco
    sinais_contaminados[var] = y

def calcular_espectro(sinal, fs):
    N_sinal = len(sinal)
    Y_fft = fft(sinal)
    freqs = fftfreq(N_sinal, 1/fs)
    metade = N_sinal // 2

    mag = np.abs(Y_fft[:metade]) / N_sinal
    return freqs[:metade], mag

freqs, mag_x = calcular_espectro(x, fs)

fig, axs = plt.subplots(4, 2, figsize=(16, 16))
fig.suptitle('Comparação entre Sinal Original $x(t)$ e Sinais Contaminados $y(t)$', fontsize=15, fontweight='bold')

t_inicio, t_fim = 1.0, 1.05
mask_t = (t >= t_inicio) & (t <= t_fim)

axs[0, 0].plot(t[mask_t], x[mask_t], color='black', lw=1)
axs[0, 0].set_title('Sinal Original $x(t)$ — Tempo (Zoom 50 ms)')
axs[0, 0].set_ylabel('Amplitude')
axs[0, 0].grid(True, alpha=0.4)

axs[0, 1].plot(freqs, mag_x, color='black', lw=1)
axs[0, 1].set_title('Sinal Original $X(f)$ — Espectro de Magnitude')
axs[0, 1].set_ylabel('Magnitude')
axs[0, 1].set_xlim(0, fs/2)
axs[0, 1].grid(True, alpha=0.4)

cores = ['tab:blue', 'tab:orange', 'tab:red']

for idx, var in enumerate(variancias):
    y = sinais_contaminados[var]
    _, mag_y = calcular_espectro(y, fs)

    axs[idx+1, 0].plot(t[mask_t], y[mask_t], color=cores[idx], lw=1)
    axs[idx+1, 0].set_title(f'Sinal Contaminado $y(t)$ ($\\sigma^2 = {var}$) — Tempo')
    axs[idx+1, 0].set_ylabel('Amplitude')
    axs[idx+1, 0].grid(True, alpha=0.4)

    axs[idx+1, 1].plot(freqs, mag_y, color=cores[idx], lw=1)
    axs[idx+1, 1].axvline(100, color='green', linestyle='--', alpha=0.8, label='Interf. 100 Hz')
    axs[idx+1, 1].axvline(2000, color='purple', linestyle='--', alpha=0.8, label='Interf. 2000 Hz')
    axs[idx+1, 1].set_title(f'Espectro $Y(f)$ ($\\sigma^2 = {var}$)')
    axs[idx+1, 1].set_ylabel('Magnitude')
    axs[idx+1, 1].set_xlim(0, fs/2)
    axs[idx+1, 1].grid(True, alpha=0.4)
    axs[idx+1, 1].legend(loc='upper right', fontsize=8.5)

axs[3, 0].set_xlabel('Tempo (s)')
axs[3, 1].set_xlabel('Frequência (Hz)')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("PLAYERS DE ÁUDIO PARA AVALIAÇÃO SUBJETIVA")
print("="*60)

print("\n1. Áudio Original x(t):")
display(Audio(x, rate=fs))

for var in variancias:
    print(f"\n2. Áudio Contaminado y(t) com variância sigma^2 = {var}:")

    display(Audio(sinais_contaminados[var], rate=fs))

##Código utilizado na questão 5-c

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)
if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64) / np.max(np.abs(x))
N = len(x)
t = np.arange(N) / fs

interf_100 = 0.05 * np.cos(200 * np.pi * t)
interf_2000 = 0.075 * np.sin(4000 * np.pi * t)

np.random.seed(42)
variancias = [1e-2, 1e-1, 1.0]

def projetar_notch(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

def projetar_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

fc1, r1 = 100.0, 0.98
fc2, r2 = 2000.0, 0.98
fc3, r3 = 3200.0, 0.65

b1, a1 = projetar_notch(fc1, fs, r1)
b2, a2 = projetar_notch(fc2, fs, r2)
b3, a3 = projetar_passa_baixas(fc3, fs, r3)

blocos = [
    ("Bloco 1: Notch (100 Hz, r=0.98)", b1, a1),
    ("Bloco 2: Notch (2000 Hz, r=0.98)", b2, a2),
    ("Bloco 3: Passa-Baixas (3200 Hz, r=0.65)", b3, a3)
]

b_total = np.convolve(np.convolve(b1, b2), b3)
a_total = np.convolve(np.convolve(a1, a2), a3)

print("=" * 80)
print(f"FUNÇÕES DE TRANSFERÊNCIA DOS BLOCOS IIR (fs = {fs} Hz)")
print("=" * 80)
for nome, b, a in blocos:
    print(f"\n{nome}:")
    print(f"  H(z) = ({b[0]:.5f} + ({b[1]:.5f})z^-1 + ({b[2]:.5f})z^-2) / "
          f"(1 + ({a[1]:.5f})z^-1 + ({a[2]:.5f})z^-2)")

fig, axs = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Projeto do Sistema de Filtragem IIR em Cascata (3 Blocos de 2ª Ordem)', fontsize=14, fontweight='bold')

angulo = np.linspace(0, 2 * np.pi, 400)
cores = ['tab:green', 'tab:purple', 'tab:orange']

for idx, (nome, b, a) in enumerate(blocos):
    w, h = signal.freqz(b, a, worN=4096, fs=fs)
    axs[0, 0].plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-10)), color=cores[idx], lw=2, label=nome)

    z_blk = np.roots(b)
    p_blk = np.roots(a)
    axs[0, 1].scatter(np.real(z_blk), np.imag(z_blk), s=100, facecolors='none', edgecolors=cores[idx], lw=2, label=f'Zeros B{idx+1}')
    axs[0, 1].scatter(np.real(p_blk), np.imag(p_blk), s=90, marker='x', color=cores[idx], lw=2, label=f'Polos B{idx+1}')

axs[0, 0].set_title('Resposta em Frequência — Blocos Individuais')
axs[0, 0].set_xlabel('Frequência (Hz)')
axs[0, 0].set_ylabel('Magnitude (dB)')
axs[0, 0].set_ylim(-50, 5)
axs[0, 0].set_xlim(0, fs/2)
axs[0, 0].grid(True, alpha=0.4)
axs[0, 0].legend(fontsize=8.5)

axs[0, 1].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5)
axs[0, 1].set_title('Diagrama de Polos e Zeros — Blocos Individuais')
axs[0, 1].set_xlabel('Parte Real')
axs[0, 1].set_ylabel('Parte Imaginária')
axs[0, 1].set_xlim(-1.3, 1.3)
axs[0, 1].set_ylim(-1.3, 1.3)
axs[0, 1].set_aspect('equal', adjustable='box')
axs[0, 1].grid(True, alpha=0.4)
axs[0, 1].legend(fontsize=8, loc='upper right')

w_tot, h_tot = signal.freqz(b_total, a_total, worN=4096, fs=fs)
axs[1, 0].plot(w_tot, 20 * np.log10(np.maximum(np.abs(h_tot), 1e-10)), color='tab:blue', lw=2.5, label='Cascata Global H(z)')
axs[1, 0].axvline(100, color='green', linestyle=':', label='Notch 100 Hz')
axs[1, 0].axvline(2000, color='purple', linestyle=':', label='Notch 2000 Hz')
axs[1, 0].axvline(3200, color='orange', linestyle=':', label='Corte PB 3200 Hz')
axs[1, 0].set_title('Resposta em Frequência — Sistema Resultante em Cascata')
axs[1, 0].set_xlabel('Frequência (Hz)')
axs[1, 0].set_ylabel('Magnitude (dB)')
axs[1, 0].set_ylim(-60, 5)
axs[1, 0].set_xlim(0, fs/2)
axs[1, 0].grid(True, alpha=0.4)
axs[1, 0].legend()

z_tot = np.roots(b_total)
p_tot = np.roots(a_total)
axs[1, 1].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5, label='Círculo Unitário')
axs[1, 1].scatter(np.real(z_tot), np.imag(z_tot), s=110, facecolors='none', edgecolors='red', lw=2, label='Zeros Totais (6)')
axs[1, 1].scatter(np.real(p_tot), np.imag(p_tot), s=100, marker='x', color='blue', lw=2, label='Polos Totais (6)')
axs[1, 1].set_title('Diagrama de Polos e Zeros — Cascata Global (6ª Ordem)')
axs[1, 1].set_xlabel('Parte Real')
axs[1, 1].set_ylabel('Parte Imaginária')
axs[1, 1].set_xlim(-1.3, 1.3)
axs[1, 1].set_ylim(-1.3, 1.3)
axs[1, 1].set_aspect('equal', adjustable='box')
axs[1, 1].grid(True, alpha=0.4)
axs[1, 1].legend(loc='upper right')

plt.tight_layout()
plt.show()

sos = np.vstack([
    np.concatenate([b1, a1]),
    np.concatenate([b2, a2]),
    np.concatenate([b3, a3])
])

fig2, axs2 = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig2.suptitle('Espectro Antes vs. Depois da Filtragem IIR em Cascata', fontsize=14, fontweight='bold')

freqs = fftfreq(N, 1/fs)[:N//2]

for idx, var in enumerate(variancias):
    ruido = np.random.normal(0, np.sqrt(var), N)
    y = x + interf_100 + interf_2000 + ruido

    y_filtrado = signal.sosfilt(sos, y)

    mag_y = np.abs(fft(y)[:N//2]) / N
    mag_f = np.abs(fft(y_filtrado)[:N//2]) / N

    axs2[idx].plot(freqs, mag_y, color='lightcoral', alpha=0.8, label=f'Contaminado $y(t)$ ($\\sigma^2={var}$)')
    axs2[idx].plot(freqs, mag_f, color='tab:blue', lw=1.2, label='Recuperado $\\hat{x}(t)$ (Pós-Cascata)')
    axs2[idx].set_title(f'Comparação Espectral para Variância $\\sigma^2 = {var}$')
    axs2[idx].set_ylabel('Magnitude')
    axs2[idx].set_xlim(0, fs/2)
    axs2[idx].grid(True, alpha=0.4)
    axs2[idx].legend(loc='upper right')

axs2[2].set_xlabel('Frequência (Hz)')
plt.tight_layout()
plt.show()

print("\n" + "="*65)
print("PLAYERS DE ÁUDIO RECUPERADO (PÓS-FILTRAGEM EM CASCATA)")
print("="*65)
for var in variancias:
    np.random.seed(42)
    y = x + interf_100 + interf_2000 + np.random.normal(0, np.sqrt(var), N)

##Código utilizado na questão 5-d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft, fftfreq
from IPython.display import Audio, display
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)
if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64) / np.max(np.abs(x))
N = len(x)
t = np.arange(N) / fs

interf_100 = 0.05 * np.cos(200 * np.pi * t)
interf_2000 = 0.075 * np.sin(4000 * np.pi * t)

def projetar_notch(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

def projetar_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

b1, a1 = projetar_notch(100.0, fs, 0.98)
b2, a2 = projetar_notch(2000.0, fs, 0.98)
b3, a3 = projetar_passa_baixas(3200.0, fs, 0.65)

sos = np.vstack([
    np.concatenate([b1, a1]),
    np.concatenate([b2, a2]),
    np.concatenate([b3, a3])
])

variancias = [1e-2, 1e-1, 1.0]
freqs = fftfreq(N, 1/fs)[:N//2]
mag_x = np.abs(fft(x)[:N//2]) / N

mask_t = (t >= 1.50) & (t <= 1.54)

fig, axs = plt.subplots(len(variancias), 2, figsize=(16, 14))
fig.suptitle('Comparação: Original $x(t)$, Contaminado $y(t)$ e Recuperado $\\hat{x}(t)$', fontsize=15, fontweight='bold')

print("=" * 80)
print(f"{'Variância (σ²)':<16} | {'SNR Antes (dB)':<18} | {'SNR Depois (dB)':<18} | {'Ganho SNR (dB)':<15}")
print("=" * 80)

for idx, var in enumerate(variancias):
    np.random.seed(42)
    ruido = np.random.normal(0, np.sqrt(var), N)
    y = x + interf_100 + interf_2000 + ruido

    x_hat = signal.sosfilt(sos, y)

    pot_sinal = np.mean(x**2)
    snr_antes = 10 * np.log10(pot_sinal / np.mean((y - x)**2))
    snr_depois = 10 * np.log10(pot_sinal / np.mean((x_hat - x)**2))
    print(f"{var:<16} | {snr_antes:<18.2f} | {snr_depois:<18.2f} | {snr_depois - snr_antes:+.2f}")

    axs[idx, 0].plot(t[mask_t], y[mask_t], color='lightcoral', alpha=0.65, lw=1, label='Contaminado $y(t)$')
    axs[idx, 0].plot(t[mask_t], x[mask_t], color='black', lw=1.8, label='Original $x(t)$')
    axs[idx, 0].plot(t[mask_t], x_hat[mask_t], color='tab:blue', lw=1.4, linestyle='--', label='Recuperado $\\hat{x}(t)$')
    axs[idx, 0].set_title(f'Domínio do Tempo (Zoom 40 ms) — $\\sigma^2 = {var}$')
    axs[idx, 0].set_ylabel('Amplitude')
    axs[idx, 0].grid(True, alpha=0.4)
    axs[idx, 0].legend(loc='upper right', fontsize=8.5)

    mag_y = np.abs(fft(y)[:N//2]) / N
    mag_xhat = np.abs(fft(x_hat)[:N//2]) / N

    axs[idx, 1].plot(freqs, mag_y, color='lightcoral', alpha=0.65, lw=1, label='Contaminado $Y(f)$')
    axs[idx, 1].plot(freqs, mag_x, color='black', lw=1.5, label='Original $X(f)$')
    axs[idx, 1].plot(freqs, mag_xhat, color='tab:blue', lw=1.2, alpha=0.9, label='Recuperado $\\hat{X}(f)$')
    axs[idx, 1].set_title(f'Domínio da Frequência (Espectro) — $\\sigma^2 = {var}$')
    axs[idx, 1].set_ylabel('Magnitude')
    axs[idx, 1].set_xlim(0, fs/2)
    axs[idx, 1].grid(True, alpha=0.4)
    axs[idx, 1].legend(loc='upper right', fontsize=8.5)

axs[-1, 0].set_xlabel('Tempo (s)')
axs[-1, 1].set_xlabel('Frequência (Hz)')
plt.tight_layout()
plt.show()

##Código utilizado na questão 5-e

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)
if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64) / np.max(np.abs(x))
N = len(x)
t = np.arange(N) / fs

interf_100 = 0.05 * np.cos(200 * np.pi * t)
interf_2000 = 0.075 * np.sin(4000 * np.pi * t)

def projetar_notch(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

def projetar_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    b = b / (np.sum(b) / np.sum(a))
    return b, a

b1, a1 = projetar_notch(100.0, fs, 0.98)
b2, a2 = projetar_notch(2000.0, fs, 0.98)
b3, a3 = projetar_passa_baixas(3200.0, fs, 0.65)

sos = np.vstack([
    np.concatenate([b1, a1]),
    np.concatenate([b2, a2]),
    np.concatenate([b3, a3])
])

variancias = [1e-2, 1e-1, 1.0]
pot_sinal = np.mean(x**2)

resultados = []

print("=" * 105)
print(f"AVALIAÇÃO QUANTITATIVA DO SISTEMA DE FILTRAGEM (Potência do Sinal Original P_x = {pot_sinal:.6f})")
print("=" * 105)
cabecalho = (f"{'Variância (σ²)':<15} | {'MSE Antes':<12} | {'MSE Depois':<12} | "
             f"{'Pot. Res. (dB)':<15} | {'SNR Antes (dB)':<15} | {'SNR Depois (dB)':<15} | {'Ganho SNR (dB)':<14}")
print(cabecalho)
print("-" * 105)

for var in variancias:
    np.random.seed(42)
    ruido_branco = np.random.normal(0, np.sqrt(var), N)

    y = x + interf_100 + interf_2000 + ruido_branco
    x_hat = signal.sosfilt(sos, y)

    erro_antes = y - x
    mse_antes = np.mean(erro_antes**2)
    snr_antes = 10 * np.log10(pot_sinal / mse_antes)

    erro_residual = x_hat - x
    pot_ruido_residual = np.mean(erro_residual**2)
    mse_depois = pot_ruido_residual
    pot_res_db = 10 * np.log10(pot_ruido_residual)
    snr_depois = 10 * np.log10(pot_sinal / pot_ruido_residual)
    ganho_snr = snr_depois - snr_antes

    resultados.append({
        'var': var,
        'mse_antes': mse_antes,
        'mse_depois': mse_depois,
        'pot_res_db': pot_res_db,
        'snr_antes': snr_antes,
        'snr_depois': snr_depois,
        'ganho_snr': ganho_snr
    })

    print(f"{var:<15} | {mse_antes:<12.6f} | {mse_depois:<12.6f} | "
          f"{pot_res_db:<15.2f} | {snr_antes:<15.2f} | {snr_depois:<15.2f} | {ganho_snr:<+14.2f}")

labels = [f"$\\sigma^2 = {v}$" for v in variancias]
x_pos = np.arange(len(variancias))
largura = 0.35

fig, axs = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Avaliação Quantitativa de Desempenho da Filtragem IIR em Cascata', fontsize=14, fontweight='bold')

snr_a = [r['snr_antes'] for r in resultados]
snr_d = [r['snr_depois'] for r in resultados]
axs[0].bar(x_pos - largura/2, snr_a, largura, label='Antes ($y(t)$)', color='lightcoral')
axs[0].bar(x_pos + largura/2, snr_d, largura, label='Depois ($\\hat{x}(t)$)', color='tab:blue')
axs[0].axhline(0, color='black', lw=0.8)
axs[0].set_title('Relação Sinal-Ruído (SNR)')
axs[0].set_ylabel('SNR (dB)')
axs[0].set_xticks(x_pos)
axs[0].set_xticklabels(labels)
axs[0].grid(True, axis='y', alpha=0.4)
axs[0].legend()

mse_a = [r['mse_antes'] for r in resultados]
mse_d = [r['mse_depois'] for r in resultados]
axs[1].bar(x_pos - largura/2, mse_a, largura, label='MSE Antes', color='lightcoral')
axs[1].bar(x_pos + largura/2, mse_d, largura, label='MSE Depois (Pot. Residual)', color='tab:green')
axs[1].set_yscale('log')
axs[1].set_title('Erro Quadrático Médio (MSE) — Escala Log')
axs[1].set_ylabel('MSE / Potência Média')
axs[1].set_xticks(x_pos)
axs[1].set_xticklabels(labels)
axs[1].grid(True, axis='y', alpha=0.4)
axs[1].legend()

ganhos = [r['ganho_snr'] for r in resultados]
bars = axs[2].bar(x_pos, ganhos, width=0.45, color='tab:purple')
axs[2].set_title('Melhoria Líquida (Ganho de SNR)')
axs[2].set_ylabel('Ganho de SNR (dB)')
axs[2].set_xticks(x_pos)
axs[2].set_xticklabels(labels)
axs[2].set_ylim(0, max(ganhos) + 1.5)
axs[2].grid(True, axis='y', alpha=0.4)

for bar in bars:
    altura = bar.get_height()
    axs[2].text(bar.get_x() + bar.get_width()/2., altura + 0.15, f'+{altura:.2f} dB', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

##Código utilizado na questão 5-f

In [ ]:
import numpy as np
from scipy.io import wavfile
from scipy import signal
from IPython.display import Audio, display
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)
if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64) / np.max(np.abs(x))
N = len(x)
t = np.arange(N) / fs

interf_100 = 0.05 * np.cos(200 * np.pi * t)
interf_2000 = 0.075 * np.sin(4000 * np.pi * t)

def projetar_notch(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / (np.sum(b) / np.sum(a)), a

def projetar_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / (np.sum(b) / np.sum(a)), a

b1, a1 = projetar_notch(100.0, fs, 0.98)
b2, a2 = projetar_notch(2000.0, fs, 0.98)
b3, a3 = projetar_passa_baixas(3200.0, fs, 0.65)

sos = np.vstack([
    np.concatenate([b1, a1]),
    np.concatenate([b2, a2]),
    np.concatenate([b3, a3])
])

print("REFERÊNCIA — Áudio Original Limpo x(t):")
display(Audio(x, rate=fs))

for var in [1e-2, 1e-1, 1.0]:
    np.random.seed(42)
    y = x + interf_100 + interf_2000 + np.random.normal(0, np.sqrt(var), N)
    x_hat = signal.sosfilt(sos, y)

    print("\n" + "="*65)
    print(f"CENÁRIO COM VARIÂNCIA DE RUÍDO BRANCO σ² = {var}")
    print("="*65)
    print(" -> Áudio Contaminado y(t):")
    display(Audio(y, rate=fs))
    print(" -> Áudio Recuperado x_hat(t):")
    display(Audio(x_hat, rate=fs))

##Código utilizado na questão 5-g

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from IPython.display import Audio, display
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2005/Dados/handel.wav"
filename = "handel.wav"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

fs, x = wavfile.read(filename)
if len(x.shape) > 1:
    x = x[:, 0]
x = x.astype(np.float64) / np.max(np.abs(x))
N = len(x)
t = np.arange(N) / fs

interf_100 = 0.05 * np.cos(200 * np.pi * t)
interf_2000 = 0.075 * np.sin(4000 * np.pi * t)

np.random.seed(42)
y = x + interf_100 + interf_2000 + np.random.normal(0, np.sqrt(1e-2), N)

def projetar_notch(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, -2 * np.cos(theta_c), 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / (np.sum(b) / np.sum(a)), a

def projetar_passa_baixas(fc, fs, r):
    theta_c = 2 * np.pi * fc / fs
    b = np.array([1.0, 2.0, 1.0])
    a = np.array([1.0, -2 * r * np.cos(theta_c), r**2])
    return b / (np.sum(b) / np.sum(a)), a

b1, a1 = projetar_notch(100.0, fs, 0.98)
b2, a2 = projetar_notch(2000.0, fs, 0.98)
b3, a3 = projetar_passa_baixas(3200.0, fs, 0.65)

sos_orig = np.vstack([
    np.concatenate([b1, a1]),
    np.concatenate([b2, a2]),
    np.concatenate([b3, a3])
])

b_dir_orig = np.convolve(np.convolve(b1, b2), b3)
a_dir_orig = np.convolve(np.convolve(a1, a2), a3)

def quantizar_vetor(vetor, b):
    max_val = np.max(np.abs(vetor))
    if max_val == 0:
        return vetor.copy()
    niveis = 2**(b - 1) - 1
    passo = max_val / niveis
    return np.round(vetor / passo) * passo

bits_lista = [2, 4, 8, 16]
pot_sinal = np.mean(x**2)

x_hat_orig = signal.sosfilt(sos_orig, y)
snr_orig = 10 * np.log10(pot_sinal / np.mean((x_hat_orig - x)**2))
w_orig, h_orig = signal.freqz(b_dir_orig, a_dir_orig, worN=4096, fs=fs)

print("=" * 100)
print(f"COMPARAÇÃO DE QUANTIZAÇÃO — CASCATA (SOS 2ª ORDEM) VS. FORMA DIRETA (6ª ORDEM) | SNR Orig: {snr_orig:.2f} dB")
print("=" * 100)
print(f"{'Bits':<6} | {'Cascata SOS (Estabilidade / max|p| / SNR)':<44} | {'Forma Direta 6ª Ordem (Estabilidade / max|p| / SNR)':<46}")
print("-" * 100)

fig, axs = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Quantização de Coeficientes: Cascata de 2ª Ordem (SOS) vs. Forma Direta (6ª Ordem)', fontsize=14, fontweight='bold')

cores = ['tab:red', 'tab:orange', 'tab:green', 'tab:purple']
angulo = np.linspace(0, 2 * np.pi, 400)

for col in [0, 1]:
    axs[0, col].plot(w_orig, 20 * np.log10(np.maximum(np.abs(h_orig), 1e-10)), 'k-', lw=2.5, label='Original (64-bit)')
    axs[1, col].plot(np.cos(angulo), np.sin(angulo), 'k--', alpha=0.5)
    axs[1, col].scatter(np.real(np.roots(b_dir_orig)), np.imag(np.roots(b_dir_orig)), s=100, facecolors='none', edgecolors='black', lw=1.5, label='Zeros Orig.')
    axs[1, col].scatter(np.real(np.roots(a_dir_orig)), np.imag(np.roots(a_dir_orig)), s=90, marker='x', color='black', lw=2, label='Polos Orig.')

audios_recuperados_sos = {}

for idx, b in enumerate(bits_lista):
    sos_q = np.zeros_like(sos_orig)
    polos_sos_q = []
    sos_valido = True

    for k in range(3):
        b_sec = quantizar_vetor(sos_orig[k, :3], b)
        a_sec = quantizar_vetor(sos_orig[k, 3:], b)

        if a_sec[0] != 0:
            b_sec = b_sec / a_sec[0]
            a_sec = a_sec / a_sec[0]
            sos_q[k, :3] = b_sec
            sos_q[k, 3:] = a_sec
            polos_sos_q.extend(np.roots(a_sec))
        else:
            sos_valido = False
            polos_sos_q.append(np.inf)

    polos_sos_q = np.array(polos_sos_q)
    max_p_sos = np.max(np.abs(polos_sos_q))
    est_sos = "ESTÁVEL" if (sos_valido and max_p_sos < 1.0) else "INSTÁVEL"

    if sos_valido:
        _, h_sos_q = signal.sosfreqz(sos_q, worN=4096, fs=fs)
        axs[0, 0].plot(w_orig, 20 * np.log10(np.maximum(np.abs(h_sos_q), 1e-10)), color=cores[idx], lw=1.5, linestyle='--', label=f'{b} bits ({est_sos})')
        axs[1, 0].scatter(np.real(polos_sos_q), np.imag(polos_sos_q), s=75, marker='+', color=cores[idx], lw=2, label=f'Polos {b}b')

    if est_sos == "ESTÁVEL":
        x_rec_sos = signal.sosfilt(sos_q, y)
        snr_sos = 10 * np.log10(pot_sinal / np.mean((x_rec_sos - x)**2))
        snr_sos_str = f"{snr_sos:.2f} dB"
        audios_recuperados_sos[b] = x_rec_sos
    else:
        snr_sos_str = "N/A (Diverge)"

    b_dir_q = quantizar_vetor(b_dir_orig, b)
    a_dir_q = quantizar_vetor(a_dir_orig, b)

    dir_valido = (a_dir_q[0] != 0)
    polos_dir_q = np.roots(a_dir_q) if dir_valido else np.array([np.inf])
    max_p_dir = np.max(np.abs(polos_dir_q))
    est_dir = "ESTÁVEL" if (dir_valido and max_p_dir < 1.0) else "INSTÁVEL"

    if dir_valido:
        _, h_dir_q = signal.freqz(b_dir_q, a_dir_q, worN=4096, fs=fs)
        axs[0, 1].plot(w_orig, 20 * np.log10(np.maximum(np.abs(h_dir_q), 1e-10)), color=cores[idx], lw=1.5, linestyle='--', label=f'{b} bits ({est_dir})')
        axs[1, 1].scatter(np.real(polos_dir_q), np.imag(polos_dir_q), s=75, marker='+', color=cores[idx], lw=2, label=f'Polos {b}b')

    if est_dir == "ESTÁVEL":
        x_rec_dir = signal.lfilter(b_dir_q, a_dir_q, y)
        snr_dir = 10 * np.log10(pot_sinal / np.mean((x_rec_dir - x)**2))
        snr_dir_str = f"{snr_dir:.2f} dB"
    else:
        snr_dir_str = "N/A (Diverge)"

    col1_txt = f"{est_sos:<9} | max|p|={max_p_sos:.4f} | SNR={snr_sos_str}"
    col2_txt = f"{est_dir:<9} | max|p|={max_p_dir:.4f} | SNR={snr_dir_str}"
    print(f"{b:<6} | {col1_txt:<44} | {col2_txt:<46}")

titulos = [
    'Resp. Frequência — Cascata de Blocos 2ª Ordem (SOS)',
    'Resp. Frequência — Implementação Direta (6ª Ordem)',
    'Polos e Zeros — Cascata de Blocos 2ª Ordem (SOS)',
    'Polos e Zeros — Implementação Direta (6ª Ordem)'
]
for i, ax in enumerate(axs.flat):
    ax.set_title(titulos[i])
    ax.grid(True, alpha=0.4)
    ax.legend(fontsize=8, loc='best')
    if i < 2:
        ax.set_xlabel('Frequência (Hz)')
        ax.set_ylabel('Magnitude (dB)')
        ax.set_ylim(-60, 15)
        ax.set_xlim(0, fs/2)
    else:
        ax.set_xlabel('Parte Real')
        ax.set_ylabel('Parte Imaginária')
        ax.set_xlim(-1.4, 1.4)
        ax.set_ylim(-1.4, 1.4)
        ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

print("\n" + "="*65)
print("PLAYERS DE ÁUDIO RECUPERADO (CASCATA SOS QUANTIZADA)")
print("="*65)
for b, audio_q in audios_recuperados_sos.items():
    print(f"\nÁudio Recuperado com b = {b} bits (Cascata SOS):")
    display(Audio(audio_q, rate=fs))